# Taiga-tundra ecotone (TTE) map on MAAP DPS

Runs `lib/tte_map.py` (Python port of the GEE TTE workflow, Montesano et al. 2020 ERL) as the DPS algorithm `run_tte_map`.

**Where to run:** MAAP Hub, Python (Pangeo) kernel, from `~/code/tte/dps`. Registering and submitting need your MAAP login, which the Hub kernel has.

Steps:
1. Register the algorithm (once per version). The code must be pushed to GitHub first, with the tag named in `algorithm_version`.
2. Submit a few test tiles, check them.
3. Submit all tiles.


In [ ]:
import os, datetime, time
import pandas as pd
import geopandas as gpd
from maap.maap import MAAP
maap = MAAP()

ALG_YML = os.path.expanduser('~/code/tte/dps/alg_tte_map/run_tte_map.yml')
ALGO_ID = 'run_tte_map'
ALGO_VERSION = 'tte_map_v1'          # git tag in github.com/pahbs/tte
USER = 'montesano'
QUEUE = 'maap-dps-worker-8gb'

## 1. Register (once per version)
The response has a link to the build job. Wait for it to finish before submitting.

In [ ]:
# maap.register_algorithm_from_yaml_file(ALG_YML).text

## 2. Inputs

Defaults reproduce the 2010 map with Hansen GFC 2018 v1.6 (2010 = treecover2000 minus 2001-2009 loss) everywhere.
The GEE original used calibrated 2 m TCC (`cal_2m_tcc`) in North America, so North America will differ from TTE_DATA_v201902v2.

For another tree cover source use `tcc_source='path'` and `tcc_paths` = comma-separated s3:// or https:// rasters/VRTs (percent cover, or set `tcc_scale`).

In [ ]:
params = {
    'tile_num': '',
    'tcc_source': 'hansen',
    'tcc_year': '2010',
    'hansen_version': 'GFC-2018-v1.6',
    'tcc_paths': 'none',
    'tcc_scale': '1',
    'tcc_resampling': 'nearest',
    'out_name': 'none',
    'gee_emulation': 'true',
    'domain_path': 's3://maap-ops-workspace/shared/montesano/tte/static/tte_domain_v201902v2.vrt',
    'tiles_gpkg': 's3://maap-ops-workspace/shared/montesano/databank/tte_tiles.gpkg',
}
IDENTIFIER = f"tte_map_hansen{params['tcc_year']}"

tiles = gpd.read_file(os.path.expanduser('~/my-public-bucket/databank/tte_tiles.gpkg'))
print(len(tiles), 'tiles')

## 3. Test tiles
Ob River (4082), Khatanga (996), Brooks Range (4219), Hudson Bay west (7026).

In [ ]:
TEST_TILES = [4082, 996, 4219, 7026]

def submit(tile_list, params, identifier=IDENTIFIER):
    rows = []
    for tn in tile_list:
        p = dict(params, tile_num=str(tn))
        r = maap.submitJob(identifier=identifier, algo_id=ALGO_ID, version=ALGO_VERSION,
                           username=USER, queue=QUEUE, **p)
        rows.append(dict(tile_num=tn, job_id=r.id, status=r.status,
                         submit_time=datetime.datetime.now().isoformat(timespec='seconds')))
    return pd.DataFrame(rows)

# jobs = submit(TEST_TILES, params); jobs

In [ ]:
def job_status(jobs):
    jobs = jobs.copy()
    jobs['status'] = [maap.getJobStatus(j) for j in jobs.job_id]
    return jobs

# job_status(jobs)

Outputs land in `~/my-private-bucket/dps_output/run_tte_map/tte_map_v1/<IDENTIFIER>/YYYY/MM/DD/...`, one COG per tile (`tte_hansen2010_NNNNNNN.tif`) plus `_stdout`/`_stderr`.

In [ ]:
import glob
outs = sorted(glob.glob(os.path.expanduser(f'~/my-private-bucket/dps_output/{ALGO_ID}/{ALGO_VERSION}/{IDENTIFIER}/**/*.tif'), recursive=True))
len(outs), outs[:5]

### Compare test tiles with the GEE product
`tte/validate_tte_vs_gee.py` resamples a Python tile onto the matching GEE tile grid and reports agreement.

In [ ]:
# !python ~/code/tte/validation/validate_tte_vs_gee.py --tile_num 4082,996 --py_dir <folder with the tifs>

## 4. All tiles in the domain
Tiles outside the TTE domain exit without writing a file. Submit in batches; 1 tile is ~1 min on the 8 GB worker.

In [ ]:
ALL_TILES = tiles.tile_num.tolist()
# jobs_all = submit(ALL_TILES, params)
# jobs_all.to_csv(os.path.expanduser(f'~/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_{IDENTIFIER}_{datetime.date.today()}.csv'), index=False)